In [3]:
# conda activate psix

import os
import sys
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

sys.path.append("code")

from modified_functions import *

In [4]:
# Load GTF

exclude = ""
gene_name = "gene_id"
gene_type = "all"
no_trim_id = False
gene_type_tag = "gene_type"
transcript_type_tag = "transcript_type"

gtf_file = "/mnt/lareaulab/reliscu/data/GENCODE/GRCh38/gencode.v46.annotation.gtf"
gtf = process_gtf(gtf_file, exclude, gene_name, no_trim_id, gene_type_tag, transcript_type_tag)
# gtf.exon_number = gtf.exon_number.astype(int)

Processing GTF file...


INFO:root:Extracted GTF attributes: ['gene_id', 'gene_type', 'gene_name', 'level', 'tag', 'transcript_id', 'transcript_type', 'transcript_name', 'transcript_support_level', 'havana_transcript', 'exon_number', 'exon_id', 'hgnc_id', 'havana_gene', 'ont', 'protein_id', 'ccdsid', 'artif_dupl']


## Get info. for all exons

In [ ]:


# Load intron table (with junctions per event)

intron_file = "/mnt/lareaulab/reliscu/data/GENCODE/GRCh38/psix_annotation/intron_file.tab.gz"
intron_table = pd.read_csv(intron_file, sep='\t', index_col=0)

intron_table_parsed = intron_table.copy()
coords = intron_table_parsed['intron'].str.split(':').str[1].str.split('-')
intron_table_parsed['intron_start'] = coords.str[0].astype(int)
intron_table_parsed['intron_end'] = coords.str[1].astype(int)

In [ ]:
# Do this ONCE before the loop

gtf_exon = gtf[gtf.feature == "exon"]
gtf_indexed = gtf_exon.set_index(['chrom', 'start', 'end']).sort_index() # for exon lookup
exons_by_transcript = {t: grp for t, grp in gtf_exon.groupby('transcript')}  # for transcript lookup


In [72]:
gtf_cds = gtf[gtf.feature == "CDS"]
cds_by_transcript = {t: grp for t, grp in gtf_cds.groupby('transcript')}  # for transcript lookup

In [136]:
gtf_sc = gtf[gtf.feature == "start_codon"]
sc_by_transcript = {t: grp for t, grp in gtf_sc.groupby('transcript')}  # for transcript lookup

In [10]:
# Compile list of all exons from cell type splicing analysis
exon_lookup = {}
for file in os.listdir("data/ctype_exons"):
    if file.endswith("exons.csv"):
        signif_exons_df = pd.read_csv(f"data/ctype_exons/{file}", index_col=0)
        signif_exons_df = signif_exons_df[~np.isnan(signif_exons_df['Oligo'])]  # keep as DataFrame, not just index
        
        for idx, row in signif_exons_df.iterrows():
            if idx not in exon_lookup:  # skip if already seen
                exon_lookup[idx] = {
                    "chr": row['chr'],
                    "exon_start": row['exon_start'],
                    "exon_end": row['exon_end']
                }

In [ ]:
exon_info = {}

# For each splicing event: save transcripts with compatible_transcripts splice junctions

for idx, val in tqdm(exon_lookup.items(), total=len(exon_lookup)):
    start, end, chrom = val['exon_start'], val['exon_end'], val['chr']
    target_exon = gtf_indexed.loc[(chrom, start, end)]
    strand = target_exon.strand.values[0]
 
    event_introns = intron_table_parsed[intron_table_parsed.event == idx]
    i1 = event_introns[event_introns.index.str.endswith("I1")]
    i2 = event_introns[event_introns.index.str.endswith("I2")]
    upstream_intron_start = i1.intron_start.values[0]
    downstream_intron_end = i2.intron_end.values[0]

    compatible_transcripts_transcripts = {}
     
    for _, exon_row in target_exon.iterrows():
        target_transcript = exon_row['transcript']
        exon_number = int(exon_row['exon_number'])
        transcript_exons = exons_by_transcript.get(target_transcript)

        # Get flanking exon boundaries (direction depends on sense)
        upstream_num, downstream_num = (exon_number + 1, exon_number - 1) if strand == "-" else (exon_number - 1, exon_number + 1) 
        te_indexed = transcript_exons.set_index(transcript_exons['exon_number'].astype(int))
        
        if upstream_num not in te_indexed.index or downstream_num not in te_indexed.index:
            continue  # exon is the last/first in this transcript

        upstream_exon_end = te_indexed.loc[upstream_num, 'end']
        downstream_exon_start = te_indexed.loc[downstream_num, 'start']

        if (upstream_intron_start == (upstream_exon_end + 1)) and (downstream_intron_end == (downstream_exon_start - 1)):
            compatible_transcripts_transcripts[target_transcript] = {
                "transcript_type": transcript_exons.transcript_type.values[0], 
                "exon_number": exon_number
            }
            
    exon_info[idx] = compatible_transcripts_transcripts

 12%|█▏        | 3438/28837 [00:39<04:51, 87.18it/s]


KeyboardInterrupt: 

In [ ]:

def is_exon_coding(exon_start, exon_end, transcript_cds_rows):
    for _, cds in transcript_cds_rows.iterrows():
        if exon_start >= cds['start'] and exon_end <= cds['end']:
            return True
    return False

In [106]:
cds_by_transcript

{'ENST00000000233':         chrom      start        end feature strand       transcript  \
 1405255  chr7  127588499  127588565     CDS      +  ENST00000000233   
 1405258  chr7  127589083  127589163     CDS      +  ENST00000000233   
 1405260  chr7  127589485  127589594     CDS      +  ENST00000000233   
 1405262  chr7  127590066  127590137     CDS      +  ENST00000000233   
 1405264  chr7  127590963  127591088     CDS      +  ENST00000000233   
 1405266  chr7  127591213  127591296     CDS      +  ENST00000000233   
 
               gene_type             gene transcript_type exon_number  frame  \
 1405255  protein_coding  ENSG00000004059  protein_coding           1      0   
 1405258  protein_coding  ENSG00000004059  protein_coding           2      2   
 1405260  protein_coding  ENSG00000004059  protein_coding           3      2   
 1405262  protein_coding  ENSG00000004059  protein_coding           4      0   
 1405264  protein_coding  ENSG00000004059  protein_coding           5      

In [134]:
from Bio import SeqIO
genome = SeqIO.to_dict(SeqIO.parse("genome.fa", "fasta"))

ModuleNotFoundError: No module named 'Bio'

In [ ]:
target_exon

feature strand       transcript       gene_type  \
chrom start     end                                                         
chr9  137023838 137023840    exon      -  ENST00000341511  protein_coding   
                137023840    exon      -  ENST00000614293  protein_coding   
                137023840    exon      -  ENST00000494046  protein_coding   
                137023840    exon      -  ENST00000476211  protein_coding   

                                      gene  transcript_type exon_number  \
chrom start     end                                                       
chr9  137023838 137023840  ENSG00000107331   protein_coding           3   
                137023840  ENSG00000107331   protein_coding           3   
                137023840  ENSG00000107331  retained_intron           3   
                137023840  ENSG00000107331  retained_intron           3   

                           frame  \
chrom start     end                
chr9  137023838 137023840      0   
                137023840      0   
                137023840      0   
                137023840      0   

                                                                         tag  \
chrom start     end                                                            
chr9  137023838 137023840  basic,Ensembl_canonical,GENCODE_Primary,MANE_S...   
                137023840  inferred_transcript_model,basic,GENCODE_Primar...   
                137023840                                        mRNA_end_NF   
                137023840                                                      

                                  protein_id  
chrom start     end                           
chr9  137023838 137023840  ENSP00000344155.6  
                137023840  ENSP00000481105.2  
                137023840                     
                137023840

In [ ]:
type()

numpy.int64

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34,
       35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49])

In [207]:
target_exon

feature strand       transcript       gene_type  \
chrom start     end                                                         
chr9  137023838 137023840    exon      -  ENST00000341511  protein_coding   
                137023840    exon      -  ENST00000614293  protein_coding   
                137023840    exon      -  ENST00000494046  protein_coding   
                137023840    exon      -  ENST00000476211  protein_coding   

                                      gene  transcript_type exon_number  \
chrom start     end                                                       
chr9  137023838 137023840  ENSG00000107331   protein_coding           3   
                137023840  ENSG00000107331   protein_coding           3   
                137023840  ENSG00000107331  retained_intron           3   
                137023840  ENSG00000107331  retained_intron           3   

                           frame  \
chrom start     end                
chr9  137023838 137023840      0   
                137023840      0   
                137023840      0   
                137023840      0   

                                                                         tag  \
chrom start     end                                                            
chr9  137023838 137023840  basic,Ensembl_canonical,GENCODE_Primary,MANE_S...   
                137023840  inferred_transcript_model,basic,GENCODE_Primar...   
                137023840                                        mRNA_end_NF   
                137023840                                                      

                                  protein_id  
chrom start     end                           
chr9  137023838 137023840  ENSP00000344155.6  
                137023840  ENSP00000481105.2  
                137023840                     
                137023840

In [ ]:
exon_info = {}
exon_coding_info = {}

# For each splicing event: save transcripts with compatible_transcripts splice junctions

for idx, val in tqdm(exon_lookup.items(), total=len(exon_lookup)):
    start, end, chrom = val['exon_start'], val['exon_end'], val['chr']
    target_exon = gtf_indexed.loc[(chrom, start, end)]
    strand = target_exon.strand.values[0]
    
    event_introns = intron_table_parsed[intron_table_parsed.event == idx]
    i1 = event_introns[event_introns.index.str.endswith("I1")]
    i2 = event_introns[event_introns.index.str.endswith("I2")]
    upstream_intron_start = i1.intron_start.values[0]
    downstream_intron_end = i2.intron_end.values[0]

    compatible_transcripts = {}

    for _, exon_row in target_exon.iterrows():
        target_transcript = exon_row['transcript']
        exon_number = int(exon_row['exon_number'])
        all_exons_in_transcript = exons_by_transcript.get(target_transcript)

        # Get flanking exon boundaries (direction depends on sense)
        upstream_num, downstream_num = (
            (exon_number + 1, exon_number - 1) if strand == "-" else (exon_number - 1, exon_number + 1)
        ) 
        te_indexed = all_exons_in_transcript.set_index(all_exons_in_transcript['exon_number'].astype(int))
        
        if upstream_num not in te_indexed.index or downstream_num not in te_indexed.index:
            continue  # if exon is the last/first exon in this particular transcript

        upstream_exon_end = te_indexed.loc[upstream_num, 'end']
        downstream_exon_start = te_indexed.loc[downstream_num, 'start']

        if (upstream_intron_start == (upstream_exon_end + 1)) and (downstream_intron_end == (downstream_exon_start - 1)):
            compatible_transcripts[target_transcript] = {
                "strand": strand,
                "transcript_type": all_exons_in_transcript.transcript_type.values[0], 
                "exon_number": exon_number,
                "tag": all_exons_in_transcript.tag.values[0]
            }
            
            # If transcript codes for a protein, get additional info:
            transcript_cds = cds_by_transcript.get(target_transcript)
            if transcript_cds is not None and is_exon_coding(start, end, transcript_cds):
                
                spliced_offset = 0      # get position of exon in spliced mRNA 
                                        # (sum up lengths of coding exons preceding target exon)
                for _, cds in transcript_cds.iterrows():
                    if cds['start'] == start and cds['end'] == end:
                        break   # reached our exon, stop
                    spliced_offset += cds['end'] - cds['start'] + 1
                
                exon_cds_start = spliced_offset
                exon_cds_end = spliced_offset + (end - start)

                this_cds_frame = (3 - (exon_cds_start % 3)) % 3
                expected_frame = transcript_cds[transcript_cds.exon_number.values.astype(int) == exon_number].frame.values[0]
                assert(expected_frame == this_cds_frame), f"Frame mismatch: {idx}"
                          
                compatible_transcripts[target_transcript]["aa_start"] = exon_cds_start // 3
                compatible_transcripts[target_transcript]["aa_end"] = exon_cds_end // 3

    exon_info[idx] = compatible_transcripts
    # if len(compatible_transcripts) > 1:
    #     break

  0%|          | 0/28837 [00:00<?, ?it/s]

## Now append exon info. to cell type exon analysis results

In [17]:
rows = []
for idx, transcripts in exon_info.items():
    rows.append({
        'exon': idx,
        'transcripts': ','.join(transcripts.keys()),
        'transcript_types': ','.join(info['transcript_type'] for info in transcripts.values()),
        'exon_numbers': ','.join(str(info['exon_number']) for info in transcripts.values())
    })

exon_info_df = pd.DataFrame(rows).set_index('exon')

In [20]:
exon_info_df.head()

,transcripts,transcript_types,exon_numbers
exon,,,
ENSG00000107331_ProteinCoding_2,"ENST00000341511,ENST00000614293,ENST0000049404...","protein_coding,protein_coding,retained_intron,...","3,3,3,3"
ENSG00000277363_other_1,ENST00000621763,protein_coding_CDS_not_defined,17
ENSG00000203485_ProteinCoding_12,"ENST00000392634,ENST00000675207,ENST0000061757...","protein_coding,protein_coding,nonsense_mediate...","22,22,21,21"
ENSG00000151150_ProteinCoding_3,"ENST00000503366,ENST00000373827,ENST0000037382...","protein_coding,protein_coding,protein_coding,p...","40,40,7,17"
ENSG00000135905_ProteinCoding_2,ENST00000535663,protein_coding,2


In [52]:
column_order = ['Gene', 'is_specific', 'specific_direction', 'exon_len', 
                'chr', 'exon_start', 'exon_end', 'transcripts', 'transcript_types', 'exon_numbers',
                'r', 'fdr', 
                'CGE Class', 'All GABAergic', 'All Neuronal',
                'Upper layer glutamatergic', 'Deep layer glutamatergic', 'Oligo', 'OPC',
                'Astro', 'Micro/PVM', 'VLMC', 'Endo', 'Peri'
                ]

In [62]:
for file in os.listdir("data/ctype_exons"):
    if file.endswith("exons.csv"):
        print(file)
        signif_exons_df = pd.read_csv(f"data/ctype_exons/{file}", index_col=0)
        signif_exons_df = signif_exons_df[~np.isnan(signif_exons_df['Oligo'])] 
        signif_exons_info = exon_info_df.merge(
            signif_exons_df, left_index=True, right_index=True,
            how='right'
        )
        rest_columns = signif_exons_info.columns[signif_exons_info.columns.str.contains("diff")].tolist() 
        new_file = file.replace('_exons.csv', '_exons_annotated.csv')
        signif_exons_info[column_order + rest_columns].to_csv(f"data/ctype_exons/annotated/{new_file}")

Oligo_exons.csv
VLMC_exons.csv
Endo_exons.csv
Deep_layer_glutamatergic_exons.csv
Astro_exons.csv
OPC_exons.csv
Micro_PVM_exons.csv
All_Neuronal_exons.csv
All_GABAergic_exons.csv
Peri_exons.csv
CGE_Class_exons.csv
Upper_layer_glutamatergic_exons.csv
